In [ ]:
# scikit-learn: tính Macro F1, Precision, Recall đồng bộ với LayoutLMv3
!pip install -q scikit-learn

from google.colab import drive
drive.mount('/content/drive')
print('✅ Đã kết nối Google Drive thành công.')

Mounted at /content/drive
✅ Đã kết nối Google Drive thành công.


In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  BƯỚC 2: CẤU HÌNH HỆ THỐNG — PATHS, HYPERPARAMS, LABEL MAP             ║
# ╠══════════════════════════════════════════════════════════════════════════╣
# ║  Tập trung toàn bộ cấu hình vào 1 cell duy nhất để dễ thay đổi.        ║
# ╚══════════════════════════════════════════════════════════════════════════╝
import os

# ── Đường dẫn gốc trên Google Drive (KHÔNG THAY ĐỔI) ──────────────────────
DRIVE_ROOT      = '/content/drive/MyDrive/LayoutLM_Project'
PROCESSED_DATA  = f'{DRIVE_ROOT}/processed_data'   # Chứa resnet_train.jsonl, resnet_val.jsonl
CHECKPOINT_DIR  = f'{DRIVE_ROOT}/checkpoints'      # Nơi lưu checkpoint 2-file cố định

# ── Đường dẫn cục bộ trên SSD Colab (đọc/ghi nhanh hơn Drive ~10x) ───────
RESNET_IMAGE_DIR = '/content/resnet_images/resnet_images'            # Thư mục ảnh sau giải nén
RESNET_ZIP       = f'{PROCESSED_DATA}/resnet_images_clean.zip'       # File zip nguồn trên Drive

# ── Tên model ──────────────────────────────────────────────────────────────
MODEL_NAME  = 'resnet50'
NUM_CLASSES = 16        # 16 lớp tài liệu chuẩn RVL-CDIP

# ── Siêu tham số Training ──────────────────────────────────────────────────
# ResNet50 dùng LR = 1e-4 (lớn hơn BERT/LayoutLMv3 vốn dùng 2e-5)
# vì ResNet50 không có cơ chế attention dễ vỡ như Transformer.
LEARNING_RATE  = 1e-4
BATCH_SIZE     = 32     # Tận dụng SSD đọc ảnh nhanh, GPU T4 xử lý tốt batch 32
NUM_EPOCHS     = 5     # Tổng số epoch muốn train (cộng dồn qua các tài khoản)
WEIGHT_DECAY   = 1e-4   # L2 regularization chống overfitting

# ── Label mapping — đồng bộ tuyệt đối với LayoutLMv3 notebook ─────────────
CLASS_TO_LABEL = {
    'letter': 0, 'form': 1, 'email': 2, 'handwritten': 3,
    'advertisement': 4, 'scientific_report': 5,
    'scientific_publication': 6, 'specification': 7,
    'file_folder': 8, 'news_article': 9, 'budget': 10,
    'invoice': 11, 'presentation': 12, 'questionnaire': 13,
    'resume': 14, 'memo': 15,
}
# Tạo list tên nhãn theo đúng thứ tự ID (dùng với sklearn classification_report)
label_list = [None] * NUM_CLASSES
for name, idx in CLASS_TO_LABEL.items():
    label_list[idx] = name

print(f'✅ Cấu hình hoàn tất. NUM_CLASSES = {NUM_CLASSES}')
print(f'   DRIVE_ROOT       : {DRIVE_ROOT}')
print(f'   PROCESSED_DATA   : {PROCESSED_DATA}')
print(f'   CHECKPOINT_DIR   : {CHECKPOINT_DIR}')
print(f'   RESNET_IMAGE_DIR : {RESNET_IMAGE_DIR}')
print(f'   LEARNING_RATE    : {LEARNING_RATE}')
print(f'   BATCH_SIZE       : {BATCH_SIZE}')
print(f'   NUM_EPOCHS       : {NUM_EPOCHS}')

✅ Cấu hình hoàn tất. NUM_CLASSES = 16
   DRIVE_ROOT       : /content/drive/MyDrive/LayoutLM_Project
   PROCESSED_DATA   : /content/drive/MyDrive/LayoutLM_Project/processed_data
   CHECKPOINT_DIR   : /content/drive/MyDrive/LayoutLM_Project/checkpoints
   RESNET_IMAGE_DIR : /content/resnet_images/resnet_images
   LEARNING_RATE    : 0.0001
   BATCH_SIZE       : 32
   NUM_EPOCHS       : 5


In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  BƯỚC 3: NẠP MODULE DỮ LIỆU & ĐÓNG BĂNG TÍNH NGẪU NHIÊN               ║
# ╠══════════════════════════════════════════════════════════════════════════╣
# ║  Thêm thư mục /code trên Drive vào sys.path để Python tìm thấy         ║
# ║  file baseline_datasets.py khi import.                                  ║
# ║                                                                          ║
# ║  seed_everything(42) PHẢI được gọi TRƯỚC KHI khởi tạo bất cứ thứ gì:  ║
# ║  model, dataloader, optimizer — để đảm bảo reproducibility tuyệt đối.  ║
# ╚══════════════════════════════════════════════════════════════════════════╝
import sys
sys.path.append(f'{DRIVE_ROOT}/code')

# Import hàm seed và factory loader từ module dùng chung của nhóm
from baseline_datasets import seed_everything, make_resnet_loaders

# ĐẶT SEED TRƯỚC MỌI THỨ — đảm bảo kết quả y hệt nhau dù đổi tài khoản Colab
seed_everything(seed=42)
print('✅ seed_everything(42) đã được gọi. Mọi phép ngẫu nhiên đã bị đóng băng.')

✅ seed_everything(42) đã được gọi. Mọi phép ngẫu nhiên đã bị đóng băng.


In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  BƯỚC 4: GIẢI NÉN ẢNH VÀO SSD COLAB                                    ║
# ╠══════════════════════════════════════════════════════════════════════════╣
# ║  BẮT BUỘC luôn tạo thư mục và giải nén đè mỗi lần khởi động session.  ║
# ║  Lý do: SSD Colab bị XÓA HOÀN TOÀN khi session kết thúc — không dùng  ║
# ║  điều kiện 'if not os.path.exists' vì sẽ bỏ qua bước này khi cần.     ║
# ║                                                                          ║
# ║  Flags unzip:                                                            ║
# ║   -o : overwrite (ghi đè file đã tồn tại, không hỏi)                   ║
# ║   -q : quiet (không in danh sách từng file ra màn hình)                ║
# ║   -d : destination (thư mục đích)                                       ║
# ╚══════════════════════════════════════════════════════════════════════════╝
import os

# Tạo thư mục đích (exist_ok=True: không báo lỗi nếu đã tồn tại)
os.makedirs(RESNET_IMAGE_DIR, exist_ok=True)
print(f'📁 Đã tạo thư mục: {RESNET_IMAGE_DIR}')

# Kiểm tra file zip nguồn có tồn tại trên Drive không
if not os.path.exists(RESNET_ZIP):
    raise FileNotFoundError(
        f'❌ Không tìm thấy file zip: {RESNET_ZIP}\n'
        f'   Hãy kiểm tra đường dẫn RESNET_ZIP ở Bước 2.'
    )

print(f'⏳ Đang giải nén từ Drive vào SSD Colab...')
print(f'   Nguồn  : {RESNET_ZIP}')
print(f'   Đích   : {RESNET_IMAGE_DIR}')

# Giải nén đè — KHÔNG dùng if/else, LUÔN chạy lệnh này
!unzip -o -q "{RESNET_ZIP}" -d "{RESNET_IMAGE_DIR}"
print('✅ Giải nén hoàn tất!')

# Kiểm tra nội dung thư mục ngay tại chỗ để xác nhận
print('\n📋 Kiểm tra nội dung thư mục ảnh (20 dòng đầu):')
!ls -l "{RESNET_IMAGE_DIR}" | head -20

📁 Đã tạo thư mục: /content/resnet_images/resnet_images
⏳ Đang giải nén từ Drive vào SSD Colab...
   Nguồn  : /content/drive/MyDrive/LayoutLM_Project/processed_data/resnet_images_clean.zip
   Đích   : /content/resnet_images/resnet_images
✅ Giải nén hoàn tất!

📋 Kiểm tra nội dung thư mục ảnh (20 dòng đầu):
total 16
drwxr-xr-x  5 root root 4096 May 23 10:06 resnet_images
drwxr-xr-x 18 root root 4096 May 23 10:06 test
drwxr-xr-x 18 root root 4096 May 23 10:05 train
drwxr-xr-x 18 root root 4096 May 23 10:06 val


In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  BƯỚC 5: TẠO DATALOADER                                                 ║
# ╠══════════════════════════════════════════════════════════════════════════╣
# ║  Gọi factory function từ baseline_datasets.py.                          ║
# ║  Train loader: augmentation online (ColorJitter, GaussianBlur, Sharp)  ║
# ║  Val loader  : chỉ ToTensor + ImageNet Normalize, KHÔNG augment.        ║
# ║                                                                          ║
# ║  Augmentation đồng bộ 100% với Cell 5 của LayoutLMv3 notebook:         ║
# ║    ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2)           ║
# ║    GaussianBlur(kernel_size=(5,5), sigma=(0.1, 2.0))                   ║
# ║    RandomAdjustSharpness(sharpness_factor=2, p=0.5)                    ║
# ╚══════════════════════════════════════════════════════════════════════════╝

loaders = make_resnet_loaders(
    data_dir     = PROCESSED_DATA,       # Chứa resnet_train.jsonl, resnet_val.jsonl
    img_base_dir = RESNET_IMAGE_DIR,     # Thư mục ảnh trên SSD vừa giải nén
    batch_size   = BATCH_SIZE,
)
train_loader = loaders['train']
val_loader   = loaders['val']

print(f'✅ DataLoader đã sẵn sàng:')
print(f'   Train : {len(train_loader):>5} batches  |  {len(train_loader.dataset):>6,} samples')
print(f'   Val   : {len(val_loader):>5} batches  |  {len(val_loader.dataset):>6,} samples')

# Kiểm tra shape của 1 batch đầu tiên để xác nhận pipeline OK
sample_batch = next(iter(train_loader))
print(f'\n   Kiểm tra shape batch:')
print(f'   pixel_values : {sample_batch["pixel_values"].shape}')  # [B, 3, 224, 224]
print(f'   label        : {sample_batch["label"].shape}')          # [B]

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


✅ DataLoader đã sẵn sàng:
   Train :   998 batches  |  31,929 samples
   Val   :   125 batches  |   3,991 samples


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()



   Kiểm tra shape batch:
   pixel_values : torch.Size([32, 3, 224, 224])
   label        : torch.Size([32])


In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  BƯỚC 6: KHỞI TẠO MÔ HÌNH RESNET50                                     ║
# ╠══════════════════════════════════════════════════════════════════════════╣
# ║  Kiến trúc: ResNet50 pretrained (ImageNet-1k) + Classification Head     ║
# ║                                                                          ║
# ║  Classification Head = nn.Linear(2048, 16)                              ║
# ║  → Đúng theo quy tắc 'Classification Head phẳng 1 tầng' của nhóm.     ║
# ║  → KHÔNG thêm Dropout hay Dense layer để đảm bảo fair comparison.      ║
# ╚══════════════════════════════════════════════════════════════════════════╝
import torch
import torch.nn as nn
from torchvision import models

# Phát hiện thiết bị — cần GPU T4 để train trong thời gian hợp lý
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'🖥️  Thiết bị: {device}')
if device.type == 'cuda':
    print(f'   GPU: {torch.cuda.get_device_name(0)}')
    print(f'   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('⚠️  Không tìm thấy GPU! Hãy vào Runtime → Change runtime type → T4 GPU')

# ── Tải ResNet50 với trọng số pretrained ImageNet-1k ──
# IMAGENET1K_V1: bản pretrained gốc, độ chính xác Top-1 = 76.1%
model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)

# ── Thay thế lớp phân loại cuối (fc layer) ──
# ResNet50 mặc định: fc = Linear(2048, 1000) — phân loại 1000 lớp ImageNet
# Ta thay bằng: fc = Linear(2048, 16)         — phân loại 16 lớp tài liệu
# KHÔNG thêm bất kỳ layer nào khác giữa backbone và fc
embedding_dim = model.fc.in_features          # = 2048 (cố định của ResNet50)
model.fc = nn.Linear(embedding_dim, NUM_CLASSES)

# Đưa toàn bộ model lên GPU
model = model.to(device)

# Thống kê tham số
total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'\n✅ Khởi tạo ResNet50 thành công:')
print(f'   Embedding dim (trước fc) : {embedding_dim}')
print(f'   Classification head      : nn.Linear({embedding_dim}, {NUM_CLASSES})')
print(f'   Tổng tham số             : {total_params:,}')
print(f'   Tham số cần train        : {trainable_params:,}')

🖥️  Thiết bị: cuda
   GPU: Tesla T4
   VRAM: 15.6 GB
Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:03<00:00, 33.9MB/s]



✅ Khởi tạo ResNet50 thành công:
   Embedding dim (trước fc) : 2048
   Classification head      : nn.Linear(2048, 16)
   Tổng tham số             : 23,540,816
   Tham số cần train        : 23,540,816


In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  BƯỚC 7A: KHỞI TẠO OPTIMIZER, SCHEDULER & LOSS                         ║
# ╚══════════════════════════════════════════════════════════════════════════╝
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

# AdamW = Adam + weight decay decoupled — chuẩn cho fine-tuning deep network
optimizer = AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

# Cosine Annealing: LR giảm dần từ LEARNING_RATE → eta_min theo đường cosine
# Giúp model hội tụ mượt mà, tránh dao động LR lớn ở epoch cuối
scheduler = CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS, eta_min=1e-6)

# CrossEntropyLoss = Softmax + NLLLoss — chuẩn cho phân loại đa lớp
criterion = nn.CrossEntropyLoss()

print(f'✅ Optimizer  : AdamW (lr={LEARNING_RATE}, weight_decay={WEIGHT_DECAY})')
print(f'   Scheduler  : CosineAnnealingLR (T_max={NUM_EPOCHS}, eta_min=1e-6)')
print(f'   Loss       : CrossEntropyLoss')

✅ Optimizer  : AdamW (lr=0.0001, weight_decay=0.0001)
   Scheduler  : CosineAnnealingLR (T_max=5, eta_min=1e-6)
   Loss       : CrossEntropyLoss


In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  BƯỚC 7B: HÀM LƯU CHECKPOINT — CHIẾN LƯỢC '2 FILE CỐ ĐỊNH'            ║
# ╠══════════════════════════════════════════════════════════════════════════╣
# ║  Duy trì đúng 2 file trên Drive cho ResNet50:                           ║
# ║  • resnet50_latest.pt : Ghi đè mỗi epoch → dùng để train tiếp          ║
# ║  • resnet50_best.pt   : Chỉ ghi đè khi Val Macro F1 tăng → dùng test  ║
# ║  torch.save() tự overwrite → dung lượng Drive không tăng theo epoch.   ║
# ╚══════════════════════════════════════════════════════════════════════════╝

def save_checkpoint_and_clean(state, is_best, checkpoint_dir, model_name):
    """
    Lưu checkpoint theo chiến lược 2-file cố định để tối ưu dung lượng Drive.

    Args:
        state          : dict chứa toàn bộ trạng thái cần phục hồi khi đổi tài khoản.
                         Gồm: epoch, model_state_dict, optimizer_state_dict,
                              scheduler_state_dict, best_val_f1, val_metrics.
        is_best        : True nếu Val Macro F1 epoch này là cao nhất từ trước đến nay.
        checkpoint_dir : Đường dẫn thư mục trên Google Drive.
        model_name     : Tên model (ở đây là 'resnet50').
    """
    import os
    os.makedirs(checkpoint_dir, exist_ok=True)

    latest_path = os.path.join(checkpoint_dir, f'{model_name}_latest.pt')
    best_path   = os.path.join(checkpoint_dir, f'{model_name}_best.pt')

    # ── File 1: Luôn ghi đè mỗi epoch để tài khoản sau train tiếp ──
    # torch.save() tự overwrite file cũ — dung lượng Drive không tăng
    torch.save(state, latest_path)
    print(f'   💾 Checkpoint nối tiếp → {latest_path}')

    # ── File 2: Chỉ cập nhật khi đạt Val Macro F1 mới cao nhất ──
    if is_best:
        torch.save(state, best_path)
        best_f1 = state['val_metrics']['macro_f1']
        print(f'   🏆 BEST MODEL! Val Macro F1 = {best_f1:.4f} → {best_path}')

print('✅ Hàm save_checkpoint_and_clean đã được định nghĩa.')

✅ Hàm save_checkpoint_and_clean đã được định nghĩa.


In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  BƯỚC 8: TỰ ĐỘNG PHỤC HỒI CHECKPOINT (Resume Training)                 ║
# ╠══════════════════════════════════════════════════════════════════════════╣
# ║  Mỗi lần chạy session mới, script kiểm tra file _latest.pt trên Drive. ║
# ║  Nếu có → nạp đầy đủ: weights + optimizer + scheduler + epoch số.      ║
# ║  Nếu không → train từ epoch 0.                                          ║
# ║                                                                          ║
# ║  Tại sao phải load cả optimizer_state_dict?                             ║
# ║  Vì AdamW lưu momentum (m, v) cho từng tham số. Nếu chỉ load weights   ║
# ║  mà bỏ optimizer state → momentum bị reset → model học lại từ đầu      ║
# ║  về mặt tối ưu, dù weights vẫn đúng.                                   ║
# ╚══════════════════════════════════════════════════════════════════════════╝
import os

latest_path = os.path.join(CHECKPOINT_DIR, f'{MODEL_NAME}_latest.pt')
start_epoch = 0       # Epoch bắt đầu (cập nhật nếu có checkpoint)
best_val_f1 = 0.0     # Mốc Macro F1 tốt nhất (cập nhật nếu có checkpoint)

if os.path.exists(latest_path):
    print(f'🔄 Tìm thấy checkpoint: {latest_path}')
    print(f'   Đang nạp trạng thái để tiếp tục training...')

    # map_location=device: đảm bảo tương thích ngay cả khi GPU thay đổi
    checkpoint = torch.load(latest_path, map_location=device)

    # Phục hồi trọng số model
    model.load_state_dict(checkpoint['model_state_dict'])

    # Phục hồi trạng thái optimizer (momentum AdamW)
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])

    # Phục hồi trạng thái scheduler (LR đang ở đâu trong chu kỳ cosine)
    if checkpoint.get('scheduler_state_dict') is not None:
        scheduler.load_state_dict(checkpoint['scheduler_state_dict'])

    # Epoch tiếp theo = epoch đã lưu (ta lưu epoch + 1 vào checkpoint)
    start_epoch = checkpoint['epoch']
    best_val_f1 = checkpoint.get('best_val_f1', 0.0)

    print(f'   ✅ Phục hồi thành công!')
    print(f'   Tiếp tục từ Epoch : {start_epoch + 1} / {NUM_EPOCHS}')
    print(f'   Best Val F1       : {best_val_f1:.4f}')
    # In lại metrics của epoch trước để so sánh
    if 'val_metrics' in checkpoint:
        m = checkpoint['val_metrics']
        print(f'   Val Epoch trước   : Loss={m["loss"]:.4f} | Acc={m["accuracy"]:.4f} | F1={m["macro_f1"]:.4f}')
else:
    print(f'🚀 Không tìm thấy checkpoint tại: {latest_path}')
    print(f'   Bắt đầu training từ đầu (Epoch 1/{NUM_EPOCHS}).')

# Cảnh báo nếu đã đủ epoch
if start_epoch >= NUM_EPOCHS:
    print(f'\n⚠️  Đã hoàn thành đủ {NUM_EPOCHS} epoch. Không cần train thêm.')
    print(f'   Để train thêm → tăng NUM_EPOCHS ở Bước 2.')

🚀 Không tìm thấy checkpoint tại: /content/drive/MyDrive/LayoutLM_Project/checkpoints/resnet50_latest.pt
   Bắt đầu training từ đầu (Epoch 1/5).


In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  BƯỚC 9: VÒNG LẶP TRAINING CHÍNH                                        ║
# ╠══════════════════════════════════════════════════════════════════════════╣
# ║  Mỗi epoch gồm 2 phase:                                                  ║
# ║  [TRAIN] model.train() → forward → loss → backward → optimizer.step()  ║
# ║  [VAL]   model.eval()  → forward → thu thập predictions → tính metrics  ║
# ║                                                                          ║
# ║  Metrics cuối epoch (đồng bộ với LayoutLMv3 — dùng scikit-learn):       ║
# ║  Loss | Accuracy | Macro F1 | Macro Precision | Macro Recall           ║
# ╚══════════════════════════════════════════════════════════════════════════╝
import numpy as np
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score
)
from tqdm.notebook import tqdm


# ─────────────────────────────────────────────────────────────────────────
# Hàm đánh giá — dùng cho Val (và sau này Test)
# ─────────────────────────────────────────────────────────────────────────
def evaluate_epoch(model, loader, criterion, device):
    """
    Chạy 1 lượt đánh giá qua toàn bộ loader (không update gradient).

    Returns:
        dict với keys: loss, accuracy, macro_f1, precision, recall
    """
    model.eval()   # Tắt Dropout, BatchNorm chuyển eval mode
    total_loss = 0.0
    all_preds  = []
    all_labels = []

    with torch.no_grad():   # Không tính gradient → tiết kiệm VRAM đáng kể
        for batch in loader:
            pixel_values = batch['pixel_values'].to(device)   # [B, 3, 224, 224]
            labels       = batch['label'].to(device)          # [B]

            outputs = model(pixel_values)            # [B, NUM_CLASSES]
            loss    = criterion(outputs, labels)

            # Cộng dồn loss (nhân với batch size để tính trung bình chính xác)
            total_loss += loss.item() * labels.size(0)

            # Lấy class có logit cao nhất làm dự đoán
            preds = torch.argmax(outputs, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    # ── Tính metrics bằng scikit-learn — đồng bộ với LayoutLMv3 ──
    avg_loss  = total_loss / len(loader.dataset)
    accuracy  = accuracy_score(all_labels, all_preds)
    # average='macro': tính F1 cho từng class rồi lấy trung bình không trọng số
    # → công bằng hơn 'weighted' khi dataset mất cân bằng
    macro_f1  = f1_score(all_labels, all_preds, average='macro', zero_division=0)
    precision = precision_score(all_labels, all_preds, average='macro', zero_division=0)
    recall    = recall_score(all_labels, all_preds, average='macro', zero_division=0)

    return {
        'loss'      : avg_loss,
        'accuracy'  : accuracy,
        'macro_f1'  : macro_f1,
        'precision' : precision,
        'recall'    : recall,
    }


# ─────────────────────────────────────────────────────────────────────────
# VÒNG LẶP TRAINING CHÍNH
# ─────────────────────────────────────────────────────────────────────────
print(f'🚀 Bắt đầu training ResNet50 từ Epoch {start_epoch + 1} / {NUM_EPOCHS}')
print('=' * 68)

for epoch in range(start_epoch, NUM_EPOCHS):

    # ════════════════════════════════════════════════════
    # PHASE 1: TRAINING
    # ════════════════════════════════════════════════════
    model.train()   # Bật Dropout, BatchNorm chuyển train mode
    train_loss   = 0.0
    train_preds  = []
    train_labels = []

    loop = tqdm(train_loader, desc=f'[Epoch {epoch+1:02d}/{NUM_EPOCHS}] TRAIN', leave=False)

    for batch in loop:
        pixel_values = batch['pixel_values'].to(device)   # [B, 3, 224, 224]
        labels       = batch['label'].to(device)          # [B]

        # ── Forward pass ──
        outputs = model(pixel_values)        # Qua backbone + fc → [B, NUM_CLASSES]
        loss    = criterion(outputs, labels) # CrossEntropy

        # ── Backward pass ──
        optimizer.zero_grad()   # Xóa gradient batch trước tránh cộng dồn sai
        loss.backward()         # Tính gradient qua toàn bộ mạng
        optimizer.step()        # Cập nhật trọng số theo gradient

        # Ghi lại để tính train metrics sau vòng lặp
        train_loss += loss.item() * labels.size(0)
        preds = torch.argmax(outputs, dim=1)
        train_preds.extend(preds.detach().cpu().numpy())
        train_labels.extend(labels.cpu().numpy())

        loop.set_postfix(loss=f'{loss.item():.4f}')   # Hiển thị loss trên tqdm

    # Cập nhật LR sau mỗi epoch (Cosine Annealing)
    scheduler.step()

    # Tính train metrics cuối epoch
    avg_train_loss = train_loss / len(train_loader.dataset)
    train_acc      = accuracy_score(train_labels, train_preds)
    train_f1       = f1_score(train_labels, train_preds, average='macro', zero_division=0)

    # ════════════════════════════════════════════════════
    # PHASE 2: VALIDATION
    # ════════════════════════════════════════════════════
    val_metrics = evaluate_epoch(model, val_loader, criterion, device)

    # ── In kết quả epoch đầy đủ ──
    current_lr = scheduler.get_last_lr()[0]
    print(f'\n{"-"*68}')
    print(f'📊 Epoch {epoch+1:02d}/{NUM_EPOCHS}   |   LR: {current_lr:.2e}')
    print(f'   TRAIN | Loss: {avg_train_loss:.4f}  |  Accuracy: {train_acc:.4f}  |  Macro F1: {train_f1:.4f}')
    print(f'   VAL   | Loss: {val_metrics["loss"]:.4f}  |  Accuracy: {val_metrics["accuracy"]:.4f}  |  Macro F1: {val_metrics["macro_f1"]:.4f}')
    print(f'          Precision: {val_metrics["precision"]:.4f}  |  Recall: {val_metrics["recall"]:.4f}')

    # ════════════════════════════════════════════════════
    # PHASE 3: LƯU CHECKPOINT
    # ════════════════════════════════════════════════════
    # Kiểm tra xem epoch này có cải thiện Val Macro F1 không
    is_best = val_metrics['macro_f1'] > best_val_f1
    if is_best:
        best_val_f1 = val_metrics['macro_f1']

    # Đóng gói TOÀN BỘ trạng thái cần thiết để phục hồi ở tài khoản khác
    checkpoint_state = {
        'epoch'               : epoch + 1,           # Epoch tiếp theo cần train
        'model_name'          : MODEL_NAME,
        'model_state_dict'    : model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'best_val_f1'         : best_val_f1,          # Mốc lịch sử để tài khoản sau so sánh
        'val_metrics'         : val_metrics,           # Metrics epoch này để theo dõi
    }

    save_checkpoint_and_clean(
        state          = checkpoint_state,
        is_best        = is_best,
        checkpoint_dir = CHECKPOINT_DIR,
        model_name     = MODEL_NAME,
    )

# In tổng kết sau khi hoàn thành tất cả epoch
print(f'\n{"="*68}')
print(f'🎉 Hoàn thành training! Best Val Macro F1 = {best_val_f1:.4f}')
print(f'   File tốt nhất : {CHECKPOINT_DIR}/{MODEL_NAME}_best.pt')

🚀 Bắt đầu training ResNet50 từ Epoch 1 / 5


[Epoch 01/5] TRAIN:   0%|          | 0/998 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()



--------------------------------------------------------------------
📊 Epoch 01/5   |   LR: 9.05e-05
   TRAIN | Loss: 1.0358  |  Accuracy: 0.6895  |  Macro F1: 0.6872
   VAL   | Loss: 0.7018  |  Accuracy: 0.7910  |  Macro F1: 0.7897
          Precision: 0.8057  |  Recall: 0.7913
   💾 Checkpoint nối tiếp → /content/drive/MyDrive/LayoutLM_Project/checkpoints/resnet50_latest.pt
   🏆 BEST MODEL! Val Macro F1 = 0.7897 → /content/drive/MyDrive/LayoutLM_Project/checkpoints/resnet50_best.pt


[Epoch 02/5] TRAIN:   0%|          | 0/998 [00:00<?, ?it/s]

...

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()



--------------------------------------------------------------------
📊 Epoch 04/5   |   LR: 1.05e-05
   TRAIN | Loss: 0.2350  |  Accuracy: 0.9298  |  Macro F1: 0.9299
   VAL   | Loss: 0.5770  |  Accuracy: 0.8411  |  Macro F1: 0.8423
          Precision: 0.8452  |  Recall: 0.8414
   💾 Checkpoint nối tiếp → /content/drive/MyDrive/LayoutLM_Project/checkpoints/resnet50_latest.pt
   🏆 BEST MODEL! Val Macro F1 = 0.8423 → /content/drive/MyDrive/LayoutLM_Project/checkpoints/resnet50_best.pt


[Epoch 05/5] TRAIN:   0%|          | 0/998 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()



--------------------------------------------------------------------
📊 Epoch 05/5   |   LR: 1.00e-06
   TRAIN | Loss: 0.1117  |  Accuracy: 0.9709  |  Macro F1: 0.9709
   VAL   | Loss: 0.5492  |  Accuracy: 0.8472  |  Macro F1: 0.8476
          Precision: 0.8492  |  Recall: 0.8473
   💾 Checkpoint nối tiếp → /content/drive/MyDrive/LayoutLM_Project/checkpoints/resnet50_latest.pt
   🏆 BEST MODEL! Val Macro F1 = 0.8476 → /content/drive/MyDrive/LayoutLM_Project/checkpoints/resnet50_best.pt

🎉 Hoàn thành training! Best Val Macro F1 = 0.8476
   File tốt nhất : /content/drive/MyDrive/LayoutLM_Project/checkpoints/resnet50_best.pt


  BẮT ĐẦU TRAINING RESNET50
  Epoch 1 → 5  |  Device: cuda
  Checkpoint dir: /content/drive/MyDrive/LayoutLM_Project/checkpoints



[Train E1]:   0%|          | 0/998 [00:00<?, ?it/s]

/tmp/ipykernel_2730/1089568919.py:20: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
/usr/local/lib/python3.12/dist-packages/torch/optim/lr_scheduler.py:1195: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scheduler.step()


KeyboardInterrupt: 

total 0
